In [2]:
# Cell 1: Scrape from original labeled dataset
import sys
import pandas as pd
import io
import requests
from pathlib import Path
from bs4 import BeautifulSoup

notebook_dir = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
project_root = notebook_dir.parent.parent.parent  # train-bert -> tests -> src -> project root
scraper_dir  = project_root / "src" / "scrapers"
models_dir   = project_root / "src" / "models"

if str(scraper_dir) not in sys.path:
    sys.path.insert(0, str(scraper_dir))

from bert_scraper import bert_scraper

try:
    import pdfplumber
    PDF_SUPPORT = True
    print("pdfplumber available")
except ImportError:
    PDF_SUPPORT = False
    print("pdfplumber not found — run: pip install pdfplumber --break-system-packages")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def scrape_pdf(url: str) -> dict:
    resp = requests.get(url, timeout=60, headers=HEADERS, allow_redirects=True)
    resp.raise_for_status()
    
    content_type = resp.headers.get("Content-Type", "")
    if "pdf" not in content_type.lower() and len(resp.content) < 1000:
        print(f"  unexpected content-type: {content_type}")
        return {"title": "", "body": ""}
    
    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        pages = []
        for page in pdf.pages[:6]:
            text = page.extract_text(x_tolerance=2, y_tolerance=3) or ""
            if text.strip():
                pages.append(text)
    
    raw = " ".join(pages)
    words = raw.split()[:300]
    title = url.split("/")[-1].replace("-", " ").replace("_", " ").replace(".pdf", "")
    return {"title": title, "body": " ".join(words)}

GOV_DOMAINS = ('fda.gov', 'cdc.gov', 'hhs.gov', 'nih.gov', 'cms.gov')

def scrape_gov(url: str) -> dict:
    resp = requests.get(url, timeout=15, headers=HEADERS)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    
    title = ""
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        title = og["content"].strip()
    elif soup.find("h1"):
        title = soup.find("h1").get_text(strip=True)

    body = ""
    for selector in ["main", "#main-content", ".content-main", 
                     "article", ".lcds-content", "#content"]:
        el = soup.select_one(selector)
        if el:
            for tag in el(["nav", "header", "footer", "script", "style"]):
                tag.decompose()
            body = el.get_text(separator=" ", strip=True)
            if len(body) > 200:
                break

    words = body.split()[:300]
    return {"title": title, "body": " ".join(words)}

def smart_scrape(url: str) -> dict:
    try:
        head = requests.head(url, timeout=10, headers=HEADERS, allow_redirects=True)
        content_type = head.headers.get("Content-Type", "").lower()
        is_pdf = "pdf" in content_type or url.lower().endswith(".pdf")
    except:
        is_pdf = url.lower().endswith(".pdf")

    if is_pdf:
        if PDF_SUPPORT:
            try:
                return scrape_pdf(url)
            except Exception as e:
                print(f"  PDF scrape failed: {e}")
                return {"title": "", "body": ""}
        else:
            return {"title": "", "body": ""}

    if any(domain in url for domain in GOV_DOMAINS):
        try:
            return scrape_gov(url)
        except Exception as e:
            print(f"  Gov scrape failed: {e}")
            return {"title": "", "body": ""}

    try:
        return bert_scraper(url)
    except Exception as e:
        print(f"  HTML scrape failed: {e}")
        return {"title": "", "body": ""}

df = pd.read_csv("Manually Collected Dataset.csv").fillna("")
print(f"Loaded {len(df)} labeled links")
print(df['subsector'].value_counts())

texts = []
for i, row in df.iterrows():
    url = row['direct_link'].strip()
    print(f"[{i+1}/{len(df)}] {url[:80]}")
    result = smart_scrape(url)
    print(f"  → body length: {len(result['body'])} chars")
    if result['body']:
        texts.append(f"{result['title']} [SEP] {result['body']}")
    else:
        texts.append("")

df['text'] = texts

df['body_length'] = df['text'].apply(
    lambda x: len(x.split('[SEP]')[1].strip()) if '[SEP]' in x else 0
)

before = len(df)
df_clean = df[df['body_length'] >= 200].drop(columns=['body_length']).reset_index(drop=True)
print(f"\nDropped {before - len(df_clean)} rows (empty or too short)")
print(f"Final clean dataset: {len(df_clean)} rows")
print(df_clean['subsector'].value_counts())

df_clean.to_csv("scraped_healthcare_data_FINAL_v2.csv", index=False)
print("\nSaved to scraped_healthcare_data_FINAL_v2.csv")

pdfplumber available
Loaded 181 labeled links
subsector
natural_disaster           31
drug_shortage              30
medical_device_shortage    30
cyber_attack               30
other                      30
none                       30
Name: count, dtype: int64
[1/181] https://www.fda.gov/drugs/drug-safety-and-availability/drug-shortages
  → body length: 1967 chars
[2/181] https://pmc.ncbi.nlm.nih.gov/articles/PMC3278171/
  → body length: 2023 chars
[3/181] https://aspe.hhs.gov/reports/shortages-three-drugs
  → body length: 1951 chars
[4/181] https://healthpolicy.duke.edu/sites/default/files/2020-02/presentation_slides__0
  → body length: 1016 chars
[5/181] https://www.mnphy.com/cover_one_David_J_Margraf_Stephen_W_Schondelmeyer
  → body length: 25 chars
[6/181] https://www.gao.gov/products/gao-25-107110
  HTML scrape failed: 403 Client Error: Forbidden for url: https://www.gao.gov/products/gao-25-107110
  → body length: 0 chars
[7/181] https://phrma.org/drug-shortages
  → body length: 

In [3]:
# Diagnostic: What URS were dropped and from where?
df_orig = pd.read_csv("Manually Collected Dataset.csv").fillna("")
df_scraped = pd.read_csv("scraped_healthcare_data_FINAL_v2.csv").fillna("")

pd.set_option('display.max_colwidth', None)
dropped = df_orig[~df_orig['direct_link'].isin(df_scraped['direct_link'])]
print(f"Dropped {len(dropped)} URLs")
print("\nDropped by subsector:")
print(dropped['subsector'].value_counts())
print("\nSample dropped URLs:")
print(dropped['direct_link'].head(20).to_string())

Dropped 87 URLs

Dropped by subsector:
subsector
drug_shortage              16
other                      16
natural_disaster           15
medical_device_shortage    14
cyber_attack               14
none                       12
Name: count, dtype: int64

Sample dropped URLs:
4                                                                       https://www.mnphy.com/cover_one_David_J_Margraf_Stephen_W_Schondelmeyer
5                                                                                                    https://www.gao.gov/products/gao-25-107110
6                                                                                                              https://phrma.org/drug-shortages
8                                                                                                 https://www.ncbi.nlm.nih.gov/books/NBK611681/
9                               https://www.pharmacytimes.com/view/addressing-drug-shortages-a-call-to-action-for-pharmacists-and-policy-makers
13 

In [15]:
# Cell 2: Config and NLI pair builder
import pandas as pd

train_df = pd.read_csv("train_split.csv").fillna("")
val_df   = pd.read_csv("val_split.csv").fillna("")

print(f"Articles — train: {len(train_df)}, val: {len(val_df)}")
print(f"Train distribution:\n{train_df['subsector'].value_counts()}")
print(f"Val distribution:\n{val_df['subsector'].value_counts()}")

HYPOTHESIS_TEMPLATE = "This healthcare news involves {}."
CANDIDATES = [
    "a drug or pharmaceutical shortage",
    "a medical device shortage",
    "a cyber attack or data breach",
    "a natural disaster affecting healthcare",
    "another healthcare disruption",
    "no significant healthcare threat",
]
SUBSECTOR_TO_CANDIDATE = {
    "drug_shortage":            "a drug or pharmaceutical shortage",
    "medical_device_shortage":  "a medical device shortage",
    "cyber_attack":             "a cyber attack or data breach",
    "natural_disaster":         "a natural disaster affecting healthcare",
    "other":                    "another healthcare disruption",
    "none":                     "no significant healthcare threat",
}
LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}

def build_nli_pairs(dataframe):
    pairs = []
    skipped = 0
    for _, row in dataframe.iterrows():
        text = str(row["text"])[:600].replace("\n", " ")
        true_candidate = SUBSECTOR_TO_CANDIDATE.get(row["subsector"].strip().lower())
        if not true_candidate:
            print(f"Unrecognized subsector: '{row['subsector']}' — skipping")
            skipped += 1
            continue
        for candidate in CANDIDATES:
            hypothesis = HYPOTHESIS_TEMPLATE.format(candidate)
            if candidate == true_candidate:
                nli_label = "entailment"
            elif true_candidate == "no significant healthcare threat":
                nli_label = "contradiction"
            elif candidate == "no significant healthcare threat":
                nli_label = "contradiction"
            else:
                nli_label = "neutral"
            pairs.append({
                "premise":    text,
                "hypothesis": hypothesis,
                "label":      LABEL2ID[nli_label],
            })
    print(f"Built {len(pairs)} NLI pairs from {len(dataframe) - skipped} articles ({skipped} skipped)")
    return pairs

Articles — train: 75, val: 19
Train distribution:
subsector
none                       14
natural_disaster           13
cyber_attack               13
medical_device_shortage    13
other                      11
drug_shortage              11
Name: count, dtype: int64
Val distribution:
subsector
none                       4
drug_shortage              3
cyber_attack               3
medical_device_shortage    3
other                      3
natural_disaster           3
Name: count, dtype: int64


In [20]:
# Cell 3: Tokenize
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_ID = "typeform/distilbert-base-uncased-mnli"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

train_pairs = build_nli_pairs(train_df)
val_pairs   = build_nli_pairs(val_df)

train_ds = Dataset.from_list(train_pairs)
val_ds   = Dataset.from_list(val_pairs)

def tokenize(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["premise", "hypothesis"])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=["premise", "hypothesis"])

print(f"Train pairs: {len(train_ds)} | Val pairs: {len(val_ds)}")
print(f"Label distribution in train: { {i: train_pairs.count({'label': i} ) for i in range(3)} }")

Built 450 NLI pairs from 75 articles (0 skipped)
Built 114 NLI pairs from 19 articles (0 skipped)


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

Train pairs: 450 | Val pairs: 114
Label distribution in train: {0: 0, 1: 0, 2: 0}


In [21]:
# Cell 4: Load model
from transformers import AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
os.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
print("Model labels:", model.config.id2label)

Using: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model labels: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [22]:
# Cell 5: Train
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report
output_path = "./healthcare_bert_production_v4"
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro":  f1_score(labels, preds, average="macro"),
        "f1_entail": f1_score(labels, preds, labels=[0], average="micro"),  # entailment only
        "accuracy":  (preds == labels).mean(),
    }

args = TrainingArguments(
    output_dir=output_path,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    metric_for_best_model="f1_macro",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
print("Saved to " + output_path)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):
/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Entail,Accuracy
1,4.159464,2.072574,0.578576,0.250000,0.710526
2,1.559001,1.481648,0.764823,0.571429,0.833333
3,1.023724,1.325068,0.739154,0.562500,0.798246
4,0.610194,1.299312,0.749399,0.606061,0.798246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):
/home/mogi/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./healthcare_bert_production_v4


In [23]:
# Cell 6: Evaluation
from sklearn.metrics import classification_report

pipeline_cls = __import__("transformers").pipeline
classifier = pipeline_cls(
    "zero-shot-classification",
    model="./healthcare_bert_production_v3",
    device=0 if torch.cuda.is_available() else -1,
)

results = []
for _, row in val_df.iterrows():
    text = str(row["text"])[:600]
    res = classifier(
        text,
        CANDIDATES,
        multi_label=False,
        hypothesis_template=HYPOTHESIS_TEMPLATE,
    )
    predicted_candidate = res["labels"][0]
    candidate_to_subsector = {v: k for k, v in SUBSECTOR_TO_CANDIDATE.items()}
    predicted_subsector = candidate_to_subsector.get(predicted_candidate, "unknown")
    
    results.append({
        "url":       row["direct_link"],
        "expected":  row["subsector"],
        "predicted": predicted_subsector,
        "correct":   row["subsector"] == predicted_subsector,
        "confidence": res["scores"][0],
    })

results_df = pd.DataFrame(results)
print(classification_report(results_df["expected"], results_df["predicted"]))
print(f"\nOverall accuracy: {results_df['correct'].mean():.1%}")
print(f"Avg confidence:   {results_df['confidence'].mean():.1%}")

misses = results_df[~results_df["correct"]]
print(f"\nMisclassified ({len(misses)}/{len(results_df)}):")
print(misses[["expected", "predicted", "confidence"]].to_string())

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

                         precision    recall  f1-score   support

           cyber_attack       1.00      1.00      1.00         3
          drug_shortage       0.25      0.33      0.29         3
medical_device_shortage       0.50      0.67      0.57         3
       natural_disaster       1.00      0.67      0.80         3
                   none       1.00      0.25      0.40         4
                  other       0.60      1.00      0.75         3

               accuracy                           0.63        19
              macro avg       0.72      0.65      0.63        19
           weighted avg       0.74      0.63      0.62        19


Overall accuracy: 63.2%
Avg confidence:   49.7%

Misclassified (7/19):
                   expected                predicted  confidence
0                      none                    other    0.431213
1             drug_shortage  medical_device_shortage    0.210273
8                      none            drug_shortage    0.272346
9   medical_dev

In [24]:
# Cell 7 — Drop-in replacement for the existing bert_classifier.py

test_text = "A hospital network took systems offline after a ransomware attack encrypted patient records."
res = classifier(test_text, CANDIDATES, multi_label=False, hypothesis_template=HYPOTHESIS_TEMPLATE)
for label, score in zip(res["labels"], res["scores"]):
    print(f"{score:.3f}  {label}")

0.945  a cyber attack or data breach
0.027  another healthcare disruption
0.010  a medical device shortage
0.008  a natural disaster affecting healthcare
0.007  a drug or pharmaceutical shortage
0.003  no significant healthcare threat


In [31]:
# Cell 8: Baseline comparison (base model vs finetunes, same val set)
from transformers import pipeline as hf_pipeline
import pandas as pd
from sklearn.metrics import classification_report

CANDIDATES = [
    "a drug or pharmaceutical shortage",
    "a medical device shortage",
    "a cyber attack or data breach",
    "a natural disaster affecting healthcare",
    "another healthcare disruption",
    "no significant healthcare threat",
]
HYPOTHESIS_TEMPLATE = "This healthcare news involves {}."
SUBSECTOR_TO_CANDIDATE = {
    "drug_shortage":            "a drug or pharmaceutical shortage",
    "medical_device_shortage":  "a medical device shortage",
    "cyber_attack":             "a cyber attack or data breach",
    "natural_disaster":         "a natural disaster affecting healthcare",
    "other":                    "another healthcare disruption",
    "none":                     "no significant healthcare threat",
}
candidate_to_subsector = {v: k for k, v in SUBSECTOR_TO_CANDIDATE.items()}

def evaluate_model(model_path, label):
    clf = hf_pipeline(
        "zero-shot-classification",
        model=model_path,
        device=0 if torch.cuda.is_available() else -1,
    )
    results = []
    for _, row in val_df.iterrows():
        text = str(row["text"])[:600]
        res = clf(text, CANDIDATES, multi_label=False, hypothesis_template=HYPOTHESIS_TEMPLATE)
        predicted = candidate_to_subsector.get(res["labels"][0], "unknown")
        results.append({
            "expected":   row["subsector"],
            "predicted":  predicted,
            "correct":    row["subsector"] == predicted,
            "confidence": res["scores"][0],
        })
    df_res = pd.DataFrame(results)
    print(f"\n{'='*20} {label} {'='*20}")
    print(classification_report(df_res["expected"], df_res["predicted"], zero_division=0))
    print(f"Overall accuracy: {df_res['correct'].mean():.1%}")
    print(f"Avg confidence:   {df_res['confidence'].mean():.1%}")
    return df_res

base_results = evaluate_model("typeform/distilbert-base-uncased-mnli",                        "BASE MODEL")
v1_results   = evaluate_model(str(models_dir / "healthcare_bert_production_v1"),              "V1 (previous finetune)")
v2_results   = evaluate_model(str(models_dir / "healthcare_bert_production_v2"),              "V2 (current finetune)")
v3_results   = evaluate_model(str(models_dir / "healthcare_bert_production_v3"),              "V3 (previous finetune)")
v4_results   = evaluate_model(str(models_dir / "healthcare_bert_production_v4"),              "V4 (previous finetune)")

print("\n===== PER-CLASS ACCURACY DELTA (vs base) =====")
for cls in sorted(val_df["subsector"].unique()):
    base_acc = (base_results[base_results["expected"] == cls]["correct"]).mean()
    v1_acc   = (v1_results[v1_results["expected"] == cls]["correct"]).mean()
    v2_acc   = (v2_results[v2_results["expected"] == cls]["correct"]).mean()
    v3_acc   = (v3_results[v3_results["expected"] == cls]["correct"]).mean()
    v4_acc   = (v4_results[v4_results["expected"] == cls]["correct"]).mean()
    print(f"{cls:<25} base: {base_acc:.0%}  v1: {v1_acc:.0%}  v2: {v2_acc:.0%}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


==================== BASE MODEL ====================
                         precision    recall  f1-score   support

           cyber_attack       1.00      0.33      0.50         3
          drug_shortage       0.50      0.33      0.40         3
medical_device_shortage       0.67      0.67      0.67         3
       natural_disaster       0.00      0.00      0.00         3
                   none       0.00      0.00      0.00         4
                  other       0.23      1.00      0.38         3

               accuracy                           0.37        19
              macro avg       0.40      0.39      0.32        19
           weighted avg       0.38      0.37      0.31        19

Overall accuracy: 36.8%
Avg confidence:   58.9%


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.



==================== V1 (previous finetune) ====================
                         precision    recall  f1-score   support

           cyber_attack       0.33      0.33      0.33         3
          drug_shortage       0.00      0.00      0.00         3
medical_device_shortage       0.00      0.00      0.00         3
       natural_disaster       1.00      0.33      0.50         3
                   none       0.00      0.00      0.00         4
                  other       0.15      0.67      0.25         3

               accuracy                           0.21        19
              macro avg       0.25      0.22      0.18        19
           weighted avg       0.23      0.21      0.17        19

Overall accuracy: 21.1%
Avg confidence:   30.7%


...


==================== V2 (current finetune) ====================
                         precision    recall  f1-score   support

           cyber_attack       1.00      1.00      1.00         3
          drug_shortage       0.50      0.67      0.57         3
medical_device_shortage       1.00      0.67      0.80         3
       natural_disaster       1.00      0.33      0.50         3
                   none       0.67      1.00      0.80         4
                  other       1.00      1.00      1.00         3

               accuracy                           0.79        19
              macro avg       0.86      0.78      0.78        19
           weighted avg       0.85      0.79      0.78        19

Overall accuracy: 78.9%
Avg confidence:   58.5%


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


==================== V3 (previous finetune) ====================
                         precision    recall  f1-score   support

           cyber_attack       1.00      1.00      1.00         3
          drug_shortage       0.25      0.33      0.29         3
medical_device_shortage       0.50      0.67      0.57         3
       natural_disaster       1.00      0.67      0.80         3
                   none       1.00      0.25      0.40         4
                  other       0.60      1.00      0.75         3

               accuracy                           0.63        19
              macro avg       0.72      0.65      0.63        19
           weighted avg       0.74      0.63      0.62        19

Overall accuracy: 63.2%
Avg confidence:   49.7%


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


==================== V4 (previous finetune) ====================
                         precision    recall  f1-score   support

           cyber_attack       1.00      1.00      1.00         3
          drug_shortage       0.25      0.33      0.29         3
medical_device_shortage       0.50      0.67      0.57         3
       natural_disaster       1.00      0.67      0.80         3
                   none       1.00      0.25      0.40         4
                  other       0.60      1.00      0.75         3

               accuracy                           0.63        19
              macro avg       0.72      0.65      0.63        19
           weighted avg       0.74      0.63      0.62        19

Overall accuracy: 63.2%
Avg confidence:   49.7%

===== PER-CLASS ACCURACY DELTA (vs base) =====
cyber_attack              base: 33%  v1: 33%  v2: 100%
drug_shortage             base: 33%  v1: 0%  v2: 67%
medical_device_shortage   base: 67%  v1: 0%  v2: 67%
natural_disaster        